In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
import lightgbm as lgb
from xgboost import XGBRegressor


In [2]:
url = 'https://raw.githubusercontent.com/shinigamiqq/EstatePricePrediction/refs/heads/main/Clear_df%2BEDA/df_final.csv'

df = pd.read_csv(url)

In [3]:
X = df.drop(['Цена', 'Город', 'Субъект РФ', 'Дата публикации', 'Цена_за_квадратный_метр', 'Цена_log', 'Цена_за_квадратный_метр_log'], axis=1)
y = df['Цена']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
from sklearn.preprocessing import StandardScaler
# Масштабирование признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
catboost = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=False
)

catboost.fit(
    X_train_scaled, y_train,
    eval_set=(X_test_scaled, y_test),
    early_stopping_rounds=200,
    plot=False
)

y_pred = catboost.predict(X_test_scaled)

print(f"MAE:  {mean_absolute_error(y_test, y_pred):,.0f} руб")
print(f"MAPE: {mean_absolute_percentage_error(y_test, y_pred)*100:.2f}%")
print(f"R2:   {r2_score(y_test, y_pred):.4f}")

MAE:  297,528 руб
MAPE: 4.08%
R2:   0.9666


In [7]:
lgb_params = {
    'objective': 'regression',
    'metric': 'mae',
    'num_leaves': 55,
    'max_depth': 8,
    'learning_rate': 0.015,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.75,
    'bagging_freq': 2,
    'lambda_l1': 0.8,
    'lambda_l2': 1.2,
    'verbosity': -1,
    'random_state': 42
}

lgb_model = lgb.LGBMRegressor(
    n_estimators=5000,
    early_stopping_rounds=200,
    **lgb_params
)

lgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
)

y_pred = lgb_model.predict(X_test_scaled)

print(f"\nРезультаты lightgbm:")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):,.0f} руб")
print(f"MAPE: {mean_absolute_percentage_error(y_test, y_pred)*100:.2f}%")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Результаты lightgbm:
MAE:  313,373 руб
MAPE: 4.25%
R²:   0.9622


In [8]:
model_xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    objective='reg:squarederror'
)

model_xgb.fit(X_train_scaled, y_train, verbose=False)

y_pred = model_xgb.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nРезультаты xgboost:")
print(f"MAE:  {mae:,.0f} руб")
print(f"MAPE: {mape*100:.2f}%")
print(f"R2:   {r2:.4f}")


Результаты xgboost:
MAE:  342,588 руб
MAPE: 4.80%
R2:   0.9627
